In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
import torch

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  
])

dataset = datasets.ImageFolder(root="data", transform=train_transform)


In [2]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

models_ckpt = {
    "vit": "google/vit-base-patch16-224",
    "deit": "facebook/deit-base-distilled-patch16-224",
    "beit": "microsoft/beit-base-patch16-224-pt22k-ft22k",
    "swinv2": "microsoft/swinv2-base-patch4-window16-256"
}

def load_model(name, num_labels=2):
    model = AutoModelForImageClassification.from_pretrained(
        models_ckpt[name],
        num_labels=num_labels
    )
    processor = AutoImageProcessor.from_pretrained(models_ckpt[name])
    return model, processor


/media/henrique/Projetos/GlaucoVision/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from torch import nn, optim
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_one_fold(model, train_loader, val_loader, epochs=10):
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs).logits
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).logits
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            gts.extend(labels.cpu().numpy())
    return preds, gts


In [4]:
import torch.nn.functional as F

def ensemble_predict(models, loader):
    all_probs = []
    for model in models:
        model.eval()
        probs = []
        with torch.no_grad():
            for imgs, _ in loader:
                imgs = imgs.to(device)
                outputs = model(imgs).logits
                probs.append(F.softmax(outputs, dim=1).cpu().numpy())
        all_probs.append(np.vstack(probs))
    
    avg_probs = np.mean(all_probs, axis=0)
    preds = np.argmax(avg_probs, axis=1)
    return preds


In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob[:,1])
    }
